## 1. Hazırlık: perf schema'sı ve trans profili

Amaç: Performance tuning verisi için `workspace.perf` schema'sını oluşturmak ve çoğaltma kararlarının varsayımlarını kontrol etmek.

Çıktı nasıl okunur:

- `max_trans_id` 10.000.000'dan küçük olmalı; kopyalarda ID'ler `trans_id + copy_no × 10.000.000` ile kaydırıldığı için çakışma olmaz.

- `min_date` / `max_date` 1993–1998 arasında olmalı (Berka formatı `YYMMDD`); her kopya 6 yıl ileri kaydırılınca dönemler üst üste binmez.

- İkinci tablo en çok işlemi olan 5 hesabı gösterir; skew için hot account bunlardan seçilir.

In [0]:
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.perf")

df_trans_bronze = spark.table("workspace.bronze.trans")

# Bronze'da her şey string; max'ı sayı olarak bulmak için int'e çevrilir
# Tarihler YYMMDD string ve hepsi 19xx olduğu için string min/max doğru sıralar
df_profile = df_trans_bronze.select(
    F.count("*").alias("rows"),
    F.max(F.col("trans_id").cast("int")).alias("max_trans_id"),
    F.min("date").alias("min_date"),
    F.max("date").alias("max_date"),
    F.countDistinct("account_id").alias("accounts"),
)
display(df_profile)

df_top_accounts = (
    df_trans_bronze.groupBy("account_id")
    .count()
    .orderBy(F.desc("count"))
    .limit(5)
)
display(df_top_accounts)

## 2. Tip dönüşümü testi

Amaç: `perf_data_generator.to_perf_trans` ile bronze trans'ı tipli hale getirmek ve dönüşümü kontrol etmek. Yazma yok.

Çıktı nasıl okunur:

- Schema: `trans_id`, `account_id` int; `trans_date` date; `amount`, `balance` decimal(12,2).

- NULL tablosu: hepsi 0 olmalı. `cast` dönüştüremediği değeri hata vermeden NULL yapar.

- Son tablo: `trans_date` 1993-01-01 – 1998-12-31; `null_k_symbol` = 535.314 (481.881 NULL + 53.433 sadece boşluk).

In [0]:
import os
import sys
import importlib

repo_root = os.path.dirname(os.getcwd())
src_path = os.path.join(repo_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

import silver_transforms
import perf_data_generator
importlib.reload(silver_transforms)
importlib.reload(perf_data_generator)

df_trans = perf_data_generator.to_perf_trans(df_trans_bronze)
df_trans.printSchema()

# Kaynakta bu kolonlarda NULL yok; burada NULL çıkarsa cast hatası demektir
null_columns = []
for column_name in ["trans_id", "account_id", "trans_date", "amount", "balance"]:
    null_columns.append(F.count_if(F.col(column_name).isNull()).alias(column_name))
display(df_trans.select(null_columns))

display(
    df_trans.select(
        F.min("trans_date").alias("min_date"),
        F.max("trans_date").alias("max_date"),
        F.count_if(F.col("k_symbol").isNull()).alias("null_k_symbol"),
    )
)

## 3. Çoğaltma testi

Amaç: `multiply_rows` ile trans'ı 5 katına çıkarmak (1 orijinal + 4 kopya) ve ID / tarih kaydırmasını kontrol etmek. Yazma yok.

Çıktı nasıl okunur:

- Kopya tablosu: her `copy_no` için 1.056.320 satır; `trans_id` her kopyada 10.000.000 kayar; tarihler 6'şar yıllık dönemlerde, üst üste binmeden (1993–1998, 1999–2004, 2005–2010, 2011–2016, 2017–2022).

- Son tablo: `rows` = `distinct_trans_id` = 5.281.600 (ID çakışması yok).

In [0]:
importlib.reload(perf_data_generator)

FACTOR = 5
ID_OFFSET = 10_000_000
SHIFT_YEARS = 6

df_trans_big = perf_data_generator.multiply_rows(
    spark, df_trans, FACTOR, "trans_id", ID_OFFSET, ["trans_date"], SHIFT_YEARS
)

df_copy_summary = (
    df_trans_big.groupBy("copy_no")
    .agg(
        F.count("*").alias("rows"),
        F.min("trans_id").alias("min_trans_id"),
        F.max("trans_id").alias("max_trans_id"),
        F.min("trans_date").alias("min_date"),
        F.max("trans_date").alias("max_date"),
    )
    .orderBy("copy_no")
)
display(df_copy_summary)

# Toplam satır = farklı ID sayısı ise kopyalar arasında ID çakışması yok
display(
    df_trans_big.select(
        F.count("*").alias("rows"),
        F.countDistinct("trans_id").alias("distinct_trans_id"),
    )
)

## 4. Skew testi

Amaç: `add_skew` ile kopyalardaki satırların ~%30'unu tek bir hot account'a (8261, bronze'da en çok işlemi olan hesap) atamak. Yazma yok.

Çıktı nasıl okunur:

- Kopya tablosu: `copy_no` 0'da `hot_rows` 675 (orijinal, dokunulmadı); 1–4'te ~317 bin, `hot_ratio` ~0,30.

- Top 5 hesap: 8261 ~1,27M satır; ikinci hesap birkaç bin satır. Join'de bir task'ın diğerlerinden çok daha uzun sürmesinin sebebi bu fark.

- Toplam satır 5.281.600 (skew satır eklemez, sadece key'i değiştirir).

In [0]:
importlib.reload(perf_data_generator)

HOT_ACCOUNT_ID = 8261
SKEW_RATIO = 0.3
SEED = 42

df_trans_skewed = perf_data_generator.add_skew(
    df_trans_big, "account_id", HOT_ACCOUNT_ID, SKEW_RATIO, SEED
)

is_hot = F.col("account_id") == HOT_ACCOUNT_ID
df_skew_summary = (
    df_trans_skewed.groupBy("copy_no")
    .agg(
        F.count("*").alias("rows"),
        F.count_if(is_hot).alias("hot_rows"),
    )
    .withColumn("hot_ratio", F.round(F.col("hot_rows") / F.col("rows"), 3))
    .orderBy("copy_no")
)
display(df_skew_summary)

df_top_accounts_skewed = (
    df_trans_skewed.groupBy("account_id")
    .count()
    .orderBy(F.desc("count"))
    .limit(5)
)
display(df_top_accounts_skewed)

display(df_trans_skewed.select(F.count("*").alias("rows")))

## 5. Tabloları yazma (factor 100)

`run` iki tabloyu `workspace.perf` altına `overwrite` ile yazar. Yukarıdaki testler factor 5 ile; yazma factor 100 ile.

Neden 100: factor 5'te (26,5 MB, tek dosya) sorgular arasında süre farkı görünmedi; EXPLAIN'de Q2'de shuffle yoktu (bkz. `11_perf_baseline`). Tarih aralığı 1993–2592 (6 yıllık kaydırma korunur; veri sentetik, gelecek tarih kabul edildi).

Nasıl okunur: `numFiles` 1'den büyük olmalı. `rows` 105.632.000, `max_trans_id` 993.682.987, tarih 1993-01-01 – 2592-12-31. `hot_rows`: `trans_big`'de 67.500 (675 × 100), `trans_skewed`'de ~31M.

In [0]:
RUN_FACTOR = 100

perf_data_generator.run(
    spark,
    factor=RUN_FACTOR,
    skew_ratio=SKEW_RATIO,
    hot_account_id=HOT_ACCOUNT_ID,
    seed=SEED,
)

df_details = None
for table_name in ["trans_big", "trans_skewed"]:
    full_name = f"{perf_data_generator.PERF_SCHEMA}.{table_name}"

    # fiziksel durum: dosya sayısı ve boyut
    df_detail = spark.sql(f"DESCRIBE DETAIL {full_name}").select(
        F.lit(table_name).alias("table_name"),
        "numFiles",
        F.round(F.col("sizeInBytes") / 1024 / 1024, 1).alias("size_mb"),
    )

    # içerik kontrolü: satır, hot account, tarih ve ID aralığı
    df_stats = spark.table(full_name).agg(
        F.count("*").alias("rows"),
        F.count_if(F.col("account_id") == HOT_ACCOUNT_ID).alias("hot_rows"),
        F.min("trans_date").alias("min_date"),
        F.max("trans_date").alias("max_date"),
        F.max("trans_id").alias("max_trans_id"),
    )

    # iki tarafta da tek satır var; crossJoin yan yana koyar
    df_detail = df_detail.crossJoin(df_stats)

    if df_details is None:
        df_details = df_detail
    else:
        df_details = df_details.unionByName(df_detail)

display(df_details)